In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
from pathlib import Path

RAW = Path("../data/raw")
seg = pd.read_csv(RAW / "swb_segments_change_2015_2025.csv", sep=";", skiprows=10,
                  thousands=".", decimal=",", nrows=18, encoding="utf-8")
seg.columns = ["segment", "change_pct", "change_abs", "jobs_2025"]
seg["segment"] = seg["segment"].str.strip()
seg = seg[seg["segment"].str.match(r"^S\d\d ")].copy()     # только 14 сегментов
seg["code"] = seg["segment"].str[:3]
seg["change_pct"] = pd.to_numeric(seg["change_pct"])

# Values read manually from IAB-Kurzbericht 5/2024, Abbildung A2 (p. 4)
subst_2022 = {"S11": 50.3, "S12": 87.9, "S13": 74.8, "S14": 42.5, "S21": 49.5, "S22": 26.5, "S23": 13.5,
              "S31": 65.9, "S32": 68.0, "S33": 66.5, "S41": 52.5, "S51": 42.5, "S52": 61.2, "S53": 45.4}
rise_2019_2022 = {"S41": ">5", "S31": ">5", "S21": ">5", "S32": ">5",
                  "S14": "1–5", "S12": "1–5", "S52": "1–5", "S33": "1–5", "S22": "1–5", "S13": "1–5", "S11": "1–5",
                  "S53": "<1", "S51": "<1", "S23": "<1"}
seg["substitution_2022"] = seg["code"].map(subst_2022)
seg["rise_2019_2022_pp"] = seg["code"].map(rise_2019_2022)
print("Correlation:", round(seg["substitution_2022"].corr(seg["change_pct"]), 2))
seg[["segment", "change_pct", "substitution_2022", "rise_2019_2022_pp"]]

Correlation: -0.69


,segment,change_pct,substitution_2022,rise_2019_2022_pp
2,"S11 Land-, Forst- und Gartenbauberufe",9,50.3,1–5
3,S12 Fertigungsberufe,-15,87.9,1–5
4,S13 Fertigungstechnische Berufe,6,74.8,1–5
5,S14 Bau- und Ausbauberufe,14,42.5,1–5
7,S21 Lebensmittel- und Gastgewerbeberufe,10,49.5,>5
8,S22 Med. u. nicht-med. Gesundheitsberufe,22,26.5,1–5
9,S23 Soziale/kulturelle Dienstleistungsberufe,39,13.5,<1
10,S31 Handelsberufe,8,65.9,>5
11,S32 Berufe in Un.-führung und -organisation,11,68.0,>5
12,S33 Unternehmensbez. Dienstleistungsberufe,17,66.5,1–5


In [2]:
fig = px.scatter(seg, x="substitution_2022", y="change_pct", size="jobs_2025", color="rise_2019_2022_pp",
                 text="code", hover_name="segment", size_max=45, height=650,
                 category_orders={"rise_2019_2022_pp": [">5", "1–5", "<1"]},
                 color_discrete_map={">5": "#d62728", "1–5": "#ff7f0e", "<1": "#1f77b4"},
                 title="The more automatable an occupation segment, the slower its employment grew (2015–2025)",
                 labels={"substitution_2022": "Substitution potential 2022, % of tasks automatable",
                         "change_pct": "Employment change 2015–2025, %",
                         "rise_2019_2022_pp": "Rise in automation potential 2019–2022 (pp)"})
slope, intercept = np.polyfit(seg["substitution_2022"], seg["change_pct"], 1)   # линия тренда
xs = np.array([10, 90])
fig.add_scatter(x=xs, y=slope * xs + intercept, mode="lines",
                line=dict(dash="dash", color="grey"), name="trend")
fig.update_traces(textposition="middle center", selector=dict(mode="markers+text"))
fig.show()
fig.write_html("../output/charts/06_automation_vs_growth.html")

## Finding 6: The more automatable, the slower the growth

- Across 14 occupation segments, substitution potential (2022) and employment growth (2015–2025) are strongly negatively correlated (r = −0.69; −0.86 without IT).
- Production work (S12): 87.9% of tasks automatable → employment −15%.
- Social & cultural services (S23): 13.5% automatable → employment +39%.
- **Exception: IT (S41)**: automation potential rose most (>5 pp, largely due to generative AI), yet employment grew fastest (+46%). Combined with the disappearing IT shortage, this may be an early signal of change.
- H5 is supported. Caveat: 14 data points, correlation ≠ causation.

Sources: IAB-Kurzbericht 5/2024 (Abb. A2); BA Strukturwandel nach Berufen. Own calculations.